In [2]:
from google.colab import drive
import os, json, time, shutil
import numpy as np
import pandas as pd

drive.mount('/content/drive')

ROOT  = "/content/drive/MyDrive/BDC_Fraud"
LOCAL = "/content/gen"
for sub in ["data", "results", "charts"]:
    os.makedirs(f"{ROOT}/{sub}", exist_ok=True)
os.makedirs(LOCAL, exist_ok=True)

print("Drive folders ready:", sorted(os.listdir(ROOT)))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive folders ready: ['charts', 'data', 'results']


In [3]:
CONFIG = {
    "seed": 42,
    "datasets": {"10k": 10_000, "100k": 100_000, "1m": 1_000_000},
    "txns_per_user": 50,
    "start_date": "2025-01-01",
    "days": 14,
    "fraud_rate": 0.015,
    "fraud_mix": {
        "high_amount": 0.40,
        "rapid_burst": 0.25,
        "impossible_travel": 0.25,
        "odd_hour": 0.10
    },
    "subtle_share": 0.25,

    "rules": {
        "zscore":   {"threshold": 3.0, "min_history": 10},
        "burst":    {"max_txns": 3, "window_sec": 120},
        "travel":   {"min_km": 500, "max_minutes": 30},
        "odd_hour": {"start_hour": 0, "end_hour": 5, "min_amount": 1500}
    },

    "streaming": {
        "replay_rates": [1000, 5000, 10000],   # events/sec
        "topic_in": "transactions",
        "topic_out": "fraud-alerts",
        "partitions": 3
    },

    "paths": {
        "root": ROOT,
        "data": f"{ROOT}/data",
        "results": f"{ROOT}/results",
        "charts": f"{ROOT}/charts"
    }
}

with open(f"{ROOT}/config.json", "w") as f:
    json.dump(CONFIG, f, indent=2)
print("config.json saved to", f"{ROOT}/config.json")

config.json saved to /content/drive/MyDrive/BDC_Fraud/config.json


In [4]:
cities = [  # name, lat, lon
    ("Mumbai", 19.0760, 72.8777), ("Pune", 18.5204, 73.8567), ("Delhi", 28.6139, 77.2090),
    ("Bengaluru", 12.9716, 77.5946), ("Hyderabad", 17.3850, 78.4867), ("Chennai", 13.0827, 80.2707),
    ("Kolkata", 22.5726, 88.3639), ("Ahmedabad", 23.0225, 72.5714), ("Jaipur", 26.9124, 75.7873),
    ("Lucknow", 26.8467, 80.9462), ("Surat", 21.1702, 72.8311), ("Nagpur", 21.1458, 79.0882),
    ("Indore", 22.7196, 75.8577), ("Bhopal", 23.2599, 77.4126), ("Kochi", 9.9312, 76.2673),
    ("Chandigarh", 30.7333, 76.7794),
]
CITY_NAMES = np.array([c[0] for c in cities])
CITY_LAT   = np.array([c[1] for c in cities])
CITY_LON   = np.array([c[2] for c in cities])

merchants_by_cat = {
    "grocery":       ["BigBasket", "DMart", "Reliance Fresh", "Spencers"],
    "fuel":          ["HP Petrol", "Indian Oil", "Shell", "BPCL"],
    "dining":        ["Zomato", "Swiggy", "Barbeque Nation", "Cafe Coffee Day"],
    "electronics":   ["Croma", "Reliance Digital", "Vijay Sales", "Apple Store"],
    "fashion":       ["Myntra", "Zara", "Westside", "Lifestyle"],
    "travel":        ["IRCTC", "MakeMyTrip", "Uber", "Ola"],
    "entertainment": ["BookMyShow", "Netflix", "PVR", "Spotify"],
    "utilities":     ["Airtel", "Jio", "Tata Power", "BSES"],
}
MERCH_NAMES = np.array([m for cat, ms in merchants_by_cat.items() for m in ms])
MERCH_CAT   = np.array([cat for cat, ms in merchants_by_cat.items() for m in ms])

HOUR_W = np.array([0.3,0.2,0.2,0.2,0.3,0.8,2,4,5,5,5,5,5,5,5,5,5,5,6,6,5,4,2,1], dtype=float)
HOUR_P = HOUR_W / HOUR_W.sum()

def haversine(lat1, lon1, lat2, lon2):
    """Great-circle distance in km (works on numpy arrays)."""
    R = 6371.0
    p1, p2 = np.radians(lat1), np.radians(lat2)
    a = np.sin((p2 - p1) / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(np.radians(lon2 - lon1) / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))

DIST     = haversine(CITY_LAT[:, None], CITY_LON[:, None], CITY_LAT[None, :], CITY_LON[None, :])
NEAREST  = np.argsort(DIST, axis=1)[:, 1:4]
FAR_MASK = DIST > 700
SUB_MASK = (DIST > 250) & (DIST < 450)

print(f"{len(CITY_NAMES)} cities, {len(MERCH_NAMES)} merchants, {len(MERCH_CAT)} categories mapped")

16 cities, 32 merchants, 32 categories mapped


In [5]:
def generate(size, cfg):
    rng = np.random.default_rng(cfg["seed"] + size)
    C, M = len(CITY_NAMES), len(MERCH_NAMES)
    n_users = max(50, size // cfg["txns_per_user"])
    days = cfg["days"]; span = days * 86400
    t0 = int(pd.Timestamp(cfg["start_date"]).timestamp())
    sub = cfg["subtle_share"]

    n_fraud = int(round(size * cfg["fraud_rate"]))
    mix = cfg["fraud_mix"]
    n_amt = int(round(n_fraud * mix["high_amount"]))
    n_trv = int(round(n_fraud * mix["impossible_travel"]))
    n_odd = int(round(n_fraud * mix["odd_hour"]))
    n_bur = n_fraud - n_amt - n_trv - n_odd
    n_norm = size - n_fraud - n_trv

    mean_amt = np.exp(rng.normal(np.log(400), 0.5, n_users))
    home     = rng.integers(0, C, n_users)
    second   = NEAREST[home, rng.integers(0, 3, n_users)]
    pref     = rng.integers(0, M, (n_users, 5))
    activity = rng.lognormal(0, 0.5, n_users); activity /= activity.sum()

    def normal_amount(u):
        a = mean_amt[u] * (1 + 0.3 * rng.standard_normal(len(u)))
        a = np.maximum(a, 0.1 * mean_amt[u])
        big = rng.random(len(u)) < 0.005
        a[big] *= rng.uniform(1.8, 2.5, big.sum())
        return a

    def normal_city(u):
        r = rng.random(len(u))
        c = np.where(r < 0.88, home[u], second[u])
        far = r > 0.995
        c[far] = rng.integers(0, C, far.sum())
        return c

    def normal_merchant(u):
        m = pref[u, rng.integers(0, 5, len(u))]
        rnd = rng.random(len(u)) > 0.8
        m[rnd] = rng.integers(0, M, rnd.sum())
        return m

    def rand_ts(n):
        day = rng.integers(0, days, n)
        hr = rng.choice(24, n, p=HOUR_P)
        return t0 + day * 86400 + hr * 3600 + rng.integers(0, 3600, n)

    parts = []
    def add(u, ts, amount, city, merch, ftype):
        parts.append(pd.DataFrame({"u": u, "ts": ts, "amount": amount,
                                   "city": city, "merch": merch, "ftype": ftype}))

    u = rng.choice(n_users, n_norm, p=activity)
    add(u, rand_ts(n_norm), normal_amount(u), normal_city(u), normal_merchant(u), "none")

    if n_amt > 0:
        u = rng.integers(0, n_users, n_amt)
        subtle = rng.random(n_amt) < sub
        mult = np.where(subtle, rng.uniform(1.5, 2.0, n_amt), rng.uniform(5, 15, n_amt))
        add(u, rand_ts(n_amt), mean_amt[u] * mult, home[u], rng.integers(0, M, n_amt), "high_amount")

    if n_bur > 0:
        ev_sub = rng.random(n_bur) < sub
        k = np.where(ev_sub, rng.integers(3, 5, n_bur), rng.integers(6, 11, n_bur))
        cs = np.cumsum(k); last = np.searchsorted(cs, n_bur)
        k = k[:last + 1].copy(); k[-1] -= (cs[last] - n_bur)
        n_ev = len(k)
        eu  = rng.integers(0, n_users, n_ev)
        est = t0 + rng.integers(0, span - 300, n_ev)
        em  = rng.integers(0, M, n_ev)
        idx = np.repeat(np.arange(n_ev), k)
        u = eu[idx]
        add(u, est[idx] + rng.integers(0, 100, len(idx)),
            mean_amt[u] * rng.uniform(0.2, 1.0, len(u)), home[u], em[idx], "rapid_burst")

    if n_trv > 0:
        u = rng.integers(0, n_users, n_trv)
        t_a = t0 + rng.integers(0, span - 3600, n_trv)
        add(u, t_a, normal_amount(u), home[u], normal_merchant(u), "none")
        subtle = rng.random(n_trv) < sub
        h = home[u]
        m = np.where(subtle[:, None], SUB_MASK[h], FAR_MASK[h])
        empty = m.sum(1) == 0
        m[empty] = FAR_MASK[h[empty]]
        dest = np.argmax(rng.random((n_trv, C)) * m, axis=1)
        gap_min = np.where(subtle, rng.uniform(5, 25, n_trv), rng.uniform(2, 15, n_trv))
        add(u, (t_a + gap_min * 60).astype(np.int64), mean_amt[u] * rng.uniform(0.5, 2.0, n_trv),
            dest, rng.integers(0, M, n_trv), "impossible_travel")

    if n_odd > 0:
        u = rng.integers(0, n_users, n_odd)
        subtle = rng.random(n_odd) < sub
        mult = np.where(subtle, rng.uniform(1.5, 2.5, n_odd), rng.uniform(4, 8, n_odd))
        ts = t0 + rng.integers(0, days, n_odd) * 86400 + rng.integers(0, 5, n_odd) * 3600 + rng.integers(0, 3600, n_odd)
        add(u, ts, mean_amt[u] * mult, home[u], rng.integers(0, M, n_odd), "odd_hour")

    df = pd.concat(parts, ignore_index=True)
    order = np.lexsort((rng.random(len(df)), df["ts"].values))
    df = df.iloc[order].reset_index(drop=True)
    n = len(df)
    ci = df["city"].values; mi = df["merch"].values
    out = pd.DataFrame({
        "txn_id": "T" + pd.Series(np.arange(1, n + 1)).astype(str).str.zfill(8),
        "user_id": "U" + df["u"].astype(str).str.zfill(5),
        "amount": df["amount"].round(2),
        "merchant": MERCH_NAMES[mi],
        "merchant_category": MERCH_CAT[mi],
        "city": CITY_NAMES[ci],
        "lat": (CITY_LAT[ci] + rng.normal(0, 0.03, n)).round(4),
        "lon": (CITY_LON[ci] + rng.normal(0, 0.03, n)).round(4),
        "timestamp": pd.to_datetime(df["ts"], unit="s").dt.strftime("%Y-%m-%d %H:%M:%S"),
        "is_fraud": (df["ftype"] != "none").astype(int),
        "fraud_type": df["ftype"],
    })
    return out

print("generate() defined")

generate() defined


In [6]:
DATA = {}
for key, size in CONFIG["datasets"].items():
    t = time.time()
    df = generate(size, CONFIG)
    assert len(df) == size, f"row count mismatch: {len(df)} vs {size}"
    local_path = f"{LOCAL}/transactions_{key}.csv"
    df.to_csv(local_path, index=False)
    shutil.copy(local_path, f"{ROOT}/data/transactions_{key}.csv")
    DATA[key] = df
    print(f"[{key:>4}] {len(df):>9,} rows  ->  {time.time()-t:5.1f}s  "
          f"({os.path.getsize(local_path)/1e6:.1f} MB)  saved to Drive")

[ 10k]    10,000 rows  ->    0.1s  (0.9 MB)  saved to Drive
[100k]   100,000 rows  ->    0.7s  (9.2 MB)  saved to Drive
[  1m] 1,000,000 rows  ->    8.3s  (92.1 MB)  saved to Drive


In [7]:
for key, df in DATA.items():
    nf = int(df["is_fraud"].sum())
    print(f"\n==================== {key.upper()} ====================")
    print(f"rows: {len(df):,} | users: {df['user_id'].nunique():,} | fraud: {nf:,} | fraud rate: {nf/len(df):.2%}")
    print(f"time range: {df['timestamp'].iloc[0]}  ->  {df['timestamp'].iloc[-1]}")
    print("fraud type breakdown:")
    print(df["fraud_type"].value_counts().to_string())

print("\nSample rows (10K dataset):")
DATA["10k"].head(8)


==================== 10K ====================
rows: 10,000 | users: 200 | fraud: 150 | fraud rate: 1.50%
time range: 2025-01-01 01:07:51  ->  2025-01-14 23:54:36
fraud type breakdown:
fraud_type
none                 9850
high_amount            60
impossible_travel      38
rapid_burst            37
odd_hour               15

==================== 100K ====================
rows: 100,000 | users: 2,000 | fraud: 1,500 | fraud rate: 1.50%
time range: 2025-01-01 00:01:22  ->  2025-01-14 23:59:22
fraud type breakdown:
fraud_type
none                 98500
high_amount            600
impossible_travel      375
rapid_burst            375
odd_hour               150

==================== 1M ====================
rows: 1,000,000 | users: 20,000 | fraud: 15,000 | fraud rate: 1.50%
time range: 2025-01-01 00:00:10  ->  2025-01-14 23:59:57
fraud type breakdown:
fraud_type
none                 985000
high_amount            6000
impossible_travel      3750
rapid_burst            3750
odd_hour             

,txn_id,user_id,amount,merchant,merchant_category,city,lat,lon,timestamp,is_fraud,fraud_type
0,T00000001,U00110,833.01,Shell,fuel,Nagpur,21.1732,79.1131,2025-01-01 01:07:51,0,none
1,T00000002,U00135,253.71,Reliance Digital,electronics,Lucknow,26.8784,80.9201,2025-01-01 01:32:51,0,none
2,T00000003,U00119,1799.19,Indian Oil,fuel,Jaipur,26.8972,75.7625,2025-01-01 02:17:54,0,none
3,T00000004,U00148,671.01,BookMyShow,entertainment,Indore,22.7460,75.8317,2025-01-01 02:24:24,0,none
4,T00000005,U00122,200.36,IRCTC,travel,Chandigarh,30.7039,76.7947,2025-01-01 02:47:45,0,none
5,T00000006,U00034,253.45,Spencers,grocery,Delhi,28.6255,77.2268,2025-01-01 03:02:42,0,none
6,T00000007,U00067,353.11,DMart,grocery,Hyderabad,17.3760,78.4739,2025-01-01 03:32:47,0,none
7,T00000008,U00159,541.21,IRCTC,travel,Kolkata,22.5757,88.3657,2025-01-01 04:17:36,0,none


In [8]:
for p in ["config.json"] + [f"data/transactions_{k}.csv" for k in CONFIG["datasets"]]:
    fp = f"{ROOT}/{p}"
    print(f"{'OK ' if os.path.exists(fp) else 'MISSING'}  {p}  ({os.path.getsize(fp)/1e6:.2f} MB)")
drive.flush_and_unmount()
print("Drive flushed.")

OK   config.json  (0.00 MB)
OK   data/transactions_10k.csv  (0.92 MB)
OK   data/transactions_100k.csv  (9.21 MB)
OK   data/transactions_1m.csv  (92.06 MB)
Drive flushed.
